In [5]:
import numpy as np
import pandas as pd

In [6]:
data_credit = pd.read_csv(r'D:\for school\business intel\movie 5000\tmdb_5000_credits.csv', low_memory=False)

In [7]:
data_movie = pd.read_csv(r'D:\for school\business intel\movie 5000\tmdb_5000_movies.csv', low_memory=False)

In [8]:
data_movie.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


## Python preprocessing for SQL Server

In [9]:
import ast
import json
from pathlib import Path

output_dir = Path(r'D:\for school\business intel\movie 5000\processed')
output_dir.mkdir(parents=True, exist_ok=True)

movies = data_movie.copy()
movies['MovieKey'] = pd.to_numeric(movies['id'], errors='coerce').astype('Int64')
movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')

numeric_cols = ['budget', 'revenue', 'vote_average', 'vote_count', 'popularity', 'runtime']
for col in numeric_cols:
    movies[col] = pd.to_numeric(movies[col], errors='coerce')

def parse_json_like(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if not isinstance(value, str) or not value.strip():
        return []
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        try:
            parsed = json.loads(value)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []

def normalize_string(value, default='Unknown'):
    if pd.isna(value) or str(value).strip() == '':
        return default
    return str(value).strip()

movies['title'] = movies['title'].fillna(movies['original_title'])
movies['original_language'] = movies['original_language'].apply(normalize_string)
movies['status'] = movies['status'].apply(normalize_string)
movies['genres_parsed'] = movies['genres'].apply(parse_json_like)
movies['DateKey'] = pd.to_numeric(movies['release_date'].dt.strftime('%Y%m%d'), errors='coerce').astype('Int64')

movies[['MovieKey', 'title', 'release_date', 'DateKey', 'genres_parsed']].head()

,MovieKey,title,release_date,DateKey,genres_parsed
0,19995,Avatar,2009-12-10,20091210,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam..."
1,285,Pirates of the Caribbean: At World's End,2007-05-19,20070519,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '..."
2,206647,Spectre,2015-10-26,20151026,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam..."
3,49026,The Dark Knight Rises,2012-07-16,20120716,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam..."
4,49529,John Carter,2012-03-07,20120307,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam..."


In [10]:
genre_expanded = movies[['MovieKey', 'genres_parsed']].explode('genres_parsed').dropna(subset=['genres_parsed']).reset_index(drop=True)
genre_details = pd.concat([genre_expanded[['MovieKey']], genre_expanded['genres_parsed'].apply(pd.Series)], axis=1)

dim_genre = (
    genre_details[['id', 'name']]
    .rename(columns={'id': 'GenreID', 'name': 'GenreName'})
    .drop_duplicates()
    .sort_values(['GenreID', 'GenreName'])
    .reset_index(drop=True)
 )

bridge_movie_genre = (
    genre_details[['MovieKey', 'id']]
    .rename(columns={'id': 'GenreID'})
    .drop_duplicates()
    .sort_values(['MovieKey', 'GenreID'])
    .reset_index(drop=True)
 )

dim_date = (
    movies.dropna(subset=['release_date'])
    .loc[:, ['DateKey', 'release_date']]
    .drop_duplicates(subset=['DateKey'])
    .sort_values('DateKey')
    .assign(
        FullDate=lambda df: df['release_date'].dt.date,
        Year=lambda df: df['release_date'].dt.year,
        Quarter=lambda df: df['release_date'].dt.quarter,
        Month=lambda df: df['release_date'].dt.month,
        MonthName=lambda df: df['release_date'].dt.month_name(),
    )
    [['DateKey', 'FullDate', 'Year', 'Quarter', 'Month', 'MonthName']]
    .reset_index(drop=True)
 )

dim_movie_info = (
    movies.loc[:, ['MovieKey', 'title', 'original_language', 'status']]
    .rename(columns={'title': 'Title', 'original_language': 'OriginalLanguage', 'status': 'Status'})
    .drop_duplicates()
    .sort_values('MovieKey')
    .reset_index(drop=True)
 )

fact_movies = (
    movies.loc[:, ['MovieKey', 'DateKey', 'budget', 'revenue', 'vote_average', 'vote_count', 'popularity', 'runtime']]
    .rename(columns={
        'budget': 'Budget',
        'revenue': 'Revenue',
        'vote_average': 'VoteAverage',
        'vote_count': 'VoteCount',
        'popularity': 'Popularity',
        'runtime': 'Runtime',
    })
    .copy()
 )
fact_movies['Profit'] = fact_movies['Revenue'] - fact_movies['Budget']

tables = {
    'DimDate': dim_date,
    'DimGenre': dim_genre,
    'DimMovieInfo': dim_movie_info,
    'BridgeMovieGenre': bridge_movie_genre,
    'FactMovies': fact_movies,
}

for table_name, frame in tables.items():
    frame.to_csv(output_dir / f'{table_name}.csv', index=False, encoding='utf-8-sig')

summary = pd.DataFrame([
    {'Table': table_name, 'Rows': len(frame), 'Columns': frame.shape[1]}
    for table_name, frame in tables.items()
]).sort_values('Table').reset_index(drop=True)

summary

,Table,Rows,Columns
0,BridgeMovieGenre,12160,2
1,DimDate,3280,6
2,DimGenre,20,2
3,DimMovieInfo,4803,4
4,FactMovies,4803,9
